# Kimi-Linear Anchor K,V Cache Experiment

Test if Kimi-Linear's KDA state can compress context.

**Supports:** 4× T4 (AWQ), 8× L4, 2× A100

## 1. Setup & Install

In [ ]:
# Install dependencies
!pip install -q vllm torch transformers

In [ ]:
# Check GPUs and auto-configure
import torch

print("GPU CHECK")
print("=" * 40)
print(f"CUDA available: {torch.cuda.is_available()}")

GPU_COUNT = torch.cuda.device_count()
print(f"GPU count: {GPU_COUNT}")

GPU_MEM = 0
for i in range(GPU_COUNT):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    GPU_MEM = max(GPU_MEM, mem)
    print(f"  GPU {i}: {name} ({mem:.1f}GB)")

# Auto-detect config
print("\nAUTO-CONFIGURATION")
print("=" * 40)

if GPU_MEM < 20:  # T4 (16GB)
    USE_QUANT = "awq"
    MAX_LEN = 8192
    print(f"Detected: T4 ({GPU_MEM:.0f}GB)")
    print(f"→ Using AWQ quantization")
elif GPU_MEM < 30:  # L4 (24GB)
    USE_QUANT = "awq"
    MAX_LEN = 16384
    print(f"Detected: L4 ({GPU_MEM:.0f}GB)")
    print(f"→ Using AWQ quantization")
else:  # A100 (40GB+)
    USE_QUANT = None
    MAX_LEN = 32768
    print(f"Detected: A100 ({GPU_MEM:.0f}GB)")
    print(f"→ No quantization needed")

print(f"→ Tensor parallel: {GPU_COUNT}")
print(f"→ Max context: {MAX_LEN}")

In [ ]:
# Install anchor connector
!git clone -q https://github.com/nydpy/anchor-kv-experiments.git /tmp/anchor-kv 2>/dev/null || true
!cd /tmp/anchor-kv && git pull -q
!bash /tmp/anchor-kv/setup_vllm.sh

## 2. Load Kimi-Linear

In [ ]:
from vllm import LLM, SamplingParams

MODEL_NAME = "moonshotai/Kimi-Linear-48B-A3B-Instruct"

print(f"Loading {MODEL_NAME}...")
print(f"  GPUs: {GPU_COUNT}")
print(f"  Quantization: {USE_QUANT or 'None'}")
print(f"  Max context: {MAX_LEN}")

llm_config = {
    "model": MODEL_NAME,
    "tensor_parallel_size": GPU_COUNT,
    "max_model_len": MAX_LEN,
    "trust_remote_code": True,
    "gpu_memory_utilization": 0.9,
}

if USE_QUANT:
    llm_config["quantization"] = USE_QUANT

llm = LLM(**llm_config)
print("\n✓ Model loaded!")

## 3. Basic Generation Test

In [ ]:
prompt = "Hello! My name is Alice and I live in Tokyo. What is my name?"

sampling_params = SamplingParams(max_tokens=30, temperature=0.0)
output = llm.generate([prompt], sampling_params)[0].outputs[0].text

print(f"Prompt: {prompt}")
print(f"Response: {output}")
print(f"\n{'✓' if 'alice' in output.lower() else '✗'} Name recognition")

## 4. Full Context vs Anchor Test

In [ ]:
# Test data
CONTEXT = """Hi, my name is Alice and I work as a software engineer at a startup in Tokyo.
I've been living here for 5 years and I really enjoy the city.
My hobbies include hiking, photography, and cooking Japanese food.
Last weekend I went to Mount Fuji and took some amazing photos."""

ANCHOR = "<alice-software-tokyo-hiking-photography/>"
INSTRUCTION = "[COMPRESSED: Anchor summarizes previous context.]\nPrevious: "
QUERY = "\n\nUser: What are your hobbies?\nAssistant:"

print(f"Context: {len(CONTEXT)} chars")
print(f"Anchor: {ANCHOR}")
print(f"Compression: {len(CONTEXT) / len(ANCHOR):.1f}x")

In [ ]:
sampling_params = SamplingParams(max_tokens=100, temperature=0.0)

# Full context
print("=" * 50)
print("1. FULL CONTEXT")
print("=" * 50)
full_output = llm.generate([CONTEXT + QUERY], sampling_params)[0].outputs[0].text
print(full_output)

In [ ]:
# Anchor only
print("=" * 50)
print("2. ANCHOR ONLY")
print("=" * 50)
anchor_output = llm.generate([INSTRUCTION + ANCHOR + QUERY], sampling_params)[0].outputs[0].text
print(anchor_output)

In [ ]:
# Compare
print("=" * 50)
print("COMPARISON")
print("=" * 50)

keywords = ['hiking', 'photography', 'cooking', 'japanese', 'fuji']
matches = 0

for kw in keywords:
    in_full = kw in full_output.lower()
    in_anchor = kw in anchor_output.lower()
    match = in_full == in_anchor
    matches += match
    print(f"  {kw}: Full={in_full}, Anchor={in_anchor} {'✓' if match else '✗'}")

print(f"\nSemantic preservation: {matches}/{len(keywords)} ({matches/len(keywords)*100:.0f}%)")

## 5. Explore KDA State

In [ ]:
print("Searching for KDA APIs...\n")

engine = llm.llm_engine
found = []

for attr in dir(engine):
    if any(x in attr.lower() for x in ['kda', 'state', 'recurrent', 'cache']):
        found.append(f"engine.{attr}")

if found:
    print("Found:")
    for f in found:
        print(f"  {f}")
else:
    print("No direct KDA APIs found on engine")
    print("May need to access model layers directly")

In [ ]:
# Try model layers
try:
    model = engine.model_executor.driver_worker.model_runner.model
    print(f"Model: {type(model).__name__}\n")
    
    for name, module in model.named_modules():
        if any(x in name.lower() for x in ['kda', 'linear_att', 'recurrent']):
            print(f"  {name}: {type(module).__name__}")
except Exception as e:
    print(f"Error: {e}")

## 6. Summary

In [ ]:
print("""
========================================
RESULTS
========================================

Check above:
1. Does anchor preserve meaning?
2. Is KDA state accessible?
3. What's the semantic preservation %?

Next steps:
- If KDA accessible → Implement save/restore
- If not → Modify vLLM or file feature request
""")